# Demonstration Experiments with Different Experiment Plans

This notebook contains example experiments demonstrating the efficiency benefit of using our proposed tree-based experiment plan in comaprison to the existing linear experiment strategies within pt.Experiment().

This notebook was used to produce the timings reported in "Trie-based Experiment Plans for Efficient IR Pipeline Experiments", by Irene Anu and Craig Macdonald, and published at ReNeuIR'26.

In [ ]:
%pip install 'pyterrier[all]'

In [ ]:
%pip install pyterrier_t5

In [ ]:
import pyterrier as pt

The experiments below are conducted on the MSMARCO v1 passage corpus, using the 43 queries of the TREC 2019 Deep Learning track. Load our BM25 retrieval index:

In [ ]:
index = pt.IndexFactory.of(pt.get_dataset('msmarco_passage').get_index('terrier_stemmed_text'), memory=False)
bm25 = pt.terrier.Retriever(index, metadata=['docno', 'text'], wmodel='BM25', verbose=True)

In [ ]:
from pyterrier_t5 import MonoT5ReRanker, DuoT5ReRanker
monoT5 = MonoT5ReRanker() # loads castorini/monot5-base-msmarco by default
duoT5 = DuoT5ReRanker() # loads castorini/duot5-base-msmarco by default
dataset = pt.get_dataset("irds:msmarco-passage/trec-dl-2019/judged")

In [ ]:
from pyterrier.measures import nDCG
pipelines = [
    bm25 % 100,
    bm25 % 100 >> monoT5,
    bm25 %100 >> monoT5 % 10 >> duoT5, 
    bm25 %100>> monoT5 % 20 >> duoT5
    ]

def one(): # no precomputation, no caching
    return pt.Experiment(
        pipelines,
        dataset.get_topics(), 
        dataset.get_qrels(),
        [nDCG@10], precompute_prefix= False,
        plan = 'linear'
    )

def two(): # precomputation, no caching
    return pt.Experiment(
        pipelines,
        dataset.get_topics(),
        dataset.get_qrels(),
        [nDCG@10],
        precompute_prefix=True, # <---- enable precomputation 
        plan = 'linear'
    )

def three(): # tree plan, no caching
    return pt.Experiment(
        pipelines,
        dataset.get_topics(),
        dataset.get_qrels(),
        [nDCG@10],
        plan='tree'  # <---- enable tree-plan 
    )

In [ ]:
%time one()

In [ ]:
%time two()

In [ ]:
%time three()

Experiments below are conducted on the MSMARCO v2 passage corpus, using the 53 queries of the TREC 2021 Deep Learning track. Load our BM25 retrieval index:

In [ ]:
index = pt.IndexFactory.of(pt.get_dataset('msmarcov2_passage').get_index('terrier_stemmed'), memory=['meta'])
bm25 = pt.terrier.Retriever(index, wmodel='BM25', verbose=True) >> pt.text.get_text(pt.get_dataset('irds:msmarco-passage-v2'), ['text'], verbose=True)
dataset_v2 = pt.get_dataset("irds:msmarco-passage-v2/trec-dl-2021/judged")

In [ ]:
from pyterrier.measures import nDCG
pipelines = [
    bm25 % 100,
    bm25 % 100 >> monoT5,
    bm25 %100 >> monoT5 % 10 >> duoT5, 
    bm25 %100>> monoT5 % 20 >> duoT5
    ]

def one(): # linear plan, no precomputation
    return pt.Experiment(
    pipelines,
    dataset_v2.get_topics(), 
    dataset_v2.get_qrels(),
    [nDCG@10], precompute_prefix= False,
    plan = 'linear'
    )

def two(): # linear plan, precomputation
    return pt.Experiment(
        pipelines,
        dataset_v2.get_topics(),
        dataset_v2.get_qrels(),
        [nDCG@10],
        precompute_prefix=True, # <---- enable precomputation 
        plan = 'linear'
    )

def three(): # tree plan
    return pt.Experiment(
        pipelines,
        dataset_v2.get_topics(),
        dataset_v2.get_qrels(),
        [nDCG@10],
        plan='tree',
    )

In [ ]:
%time one()

In [ ]:
%time two()

In [ ]:
%time three()